In [1]:
from langgraph.graph import StateGraph , START, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [2]:
# define the state
class CrushState(TypedDict):
    input : str
    step1 : str
    step2 : str
    step3 : str

# define the steps
def step1(State : CrushState) -> CrushState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": State["input"]}

def step2(State: CrushState)-> CrushState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(1000) 
    return {"step2": "done"}

def step3(State: CrushState)-> CrushState:
    print("✅ Step 3 executed")
    return {"step3": 'done'}

In [4]:
# build the graph
graph = StateGraph(CrushState)

# add node
graph.add_node('step1',step1)
graph.add_node('step2',step2)
graph.add_node('step3',step3)

# add edge
graph.add_edge(START,'step1')
graph.add_edge('step2','step3')
graph.add_edge('step3',END)

checkpointer = InMemorySaver()
workflow = graph.compile(checkpointer=checkpointer)
workflow

ValueError: Failed to reach https://mermaid.ink API while trying to render your graph after 1 retries. To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`

In [ ]:
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

In [ ]:
print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)

In [ ]:
list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))